# Several RV instruments: one zero point and one jitter per instrument

Radial velocities (RVs) of one star often come from several instruments. Each instrument has its own **zero point** (a constant added to all its RVs) and its own **jitter** (extra noise, not in the quoted errors).

No function of orblet takes several instruments. You do not need one. The pattern uses the atoms as they are:

1. **Linear layer.** The RV model at a fixed orbit shape $(P, e, \tau)$ is linear in $(\gamma, C, S)$, with $C = K\cos\omega$ and $S = K\sin\omega$. Add **one 0/1 column per instrument other than the reference instrument**. Its amplitude is the zero point of that instrument minus the zero point of the reference.
2. **Jitter.** The solve takes one error per row. The jitter of an instrument goes into the errors of its rows: $\sqrt{\sigma^2 + s^2}$.
3. **Likelihood.** `loglike` takes one zero point and one jitter. The likelihood of several instruments is one call per instrument, added.

The data here are synthetic. Units: times in **MJD**, periods in **days** (orblet's RV design takes Keplerian **years**), velocities in **km/s**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from orblet.constants import DAYS_PER_KEPLER_YEAR
from orblet.design import RvOrbitBuilder, concat_columns
from orblet.likelihood import loglike
from orblet.model import rv_model
from orblet.solve.rv import linear_solve_rv, recover_K, recover_omega

## 1. Three synthetic instruments

One orbit: $P = 4000$ d, $e = 0.70$. Three instruments observe it:

| instrument | rows | when | quoted error | true zero point | true jitter |
|---|---:|---|---:|---:|---:|
| `gaia` (the reference) | 17 | the first 1800 d | 1.5 | 20.0 | 0.0 |
| `B` | 25 | overlaps `gaia`, then 2400 d more | 0.2 | 21.5 | 0.3 |
| `C` | 12 | after `gaia` | 0.4 | 19.2 | 0.6 |

What is real: orblet's RV model. What is a stand-in: the orbit, the epochs, the errors and the noise.

In [ ]:
EPOCH_REF_MJD = 57000.0
# the true orbit, in the keywords of rv_model and loglike
TRUE_ORBIT = dict(period_yr=4000.0 / DAYS_PER_KEPLER_YEAR, ecc=0.70, omega_rad=1.2, tau=0.35,
                  mass_msun=30.0, M_msun=31.0)
NAMES = ["gaia", "B", "C"]                      # NAMES[0] is the reference instrument
# three tables from the instrument name to a number in km/s
TRUE_ZERO_POINT = {"gaia": 20.0, "B": 21.5, "C": 19.2}
TRUE_JITTER = {"gaia": 0.0, "B": 0.3, "C": 0.6}
QUOTED_ERROR = {"gaia": 1.5, "B": 0.2, "C": 0.4}

rng_times = np.random.default_rng(11)           # seed 11: the epochs
rng_noise = np.random.default_rng(12)           # seed 12: the noise
TIMES = {
    "gaia": np.sort(rng_times.uniform(57000.0, 58800.0, 17)),
    "B": np.sort(rng_times.uniform(58200.0, 61200.0, 25)),
    "C": np.sort(rng_times.uniform(60000.0, 61500.0, 12)),
}

# DATA: a table from the instrument name to its times, RVs and quoted errors
DATA = {}
for name in NAMES:
    model = rv_model(TIMES[name], offset_kms=TRUE_ZERO_POINT[name],
                     epoch_ref_mjd=EPOCH_REF_MJD, **TRUE_ORBIT)
    noise_sigma = np.hypot(QUOTED_ERROR[name], TRUE_JITTER[name])   # quoted error and jitter together
    noise = rng_noise.normal(0.0, noise_sigma, model.size)
    DATA[name] = dict(t=TIMES[name], rv=model + noise, err=np.full(model.size, QUOTED_ERROR[name]))

fig, ax = plt.subplots(figsize=(9, 3.5))
for name in NAMES:
    ax.errorbar(DATA[name]["t"], DATA[name]["rv"], yerr=DATA[name]["err"], fmt="o", ms=4, label=name)
ax.set_xlabel("time (MJD)")
ax.set_ylabel("RV (km/s)")
ax.legend()
plt.show()

## 2. The design matrix: three columns, plus one 0/1 column per extra instrument

Stack the rows of all instruments into one table. Keep, for each row, the number of its instrument.

In [ ]:
# step 1, stack: one row per RV; instrument_number is 0 for "gaia", 1 for "B", 2 for "C"
t_all = np.concatenate([DATA[name]["t"] for name in NAMES])
rv_all = np.concatenate([DATA[name]["rv"] for name in NAMES])
err_all = np.concatenate([DATA[name]["err"] for name in NAMES])
number_parts = []
for number, name in enumerate(NAMES):           # number counts the instruments
    number_parts.append(np.full(DATA[name]["t"].size, number))
instrument_number = np.concatenate(number_parts)

# step 2, the jitter of each row's instrument, and the error the solve will use.
# The linear layer holds the jitter FIXED. Here it is set to the true value; section 5 fits it.
jitter_all = np.array([TRUE_JITTER[NAMES[number]] for number in instrument_number])
sigma_all = np.hypot(err_all, jitter_all)


def design_matrix(period_days, ecc, tau, own_zero_points):
    # [gamma, C, S] at one shape; with own_zero_points, one 0/1 column for "B" and one for "C".
    X = RvOrbitBuilder().build(t_all, period_yr=period_days / DAYS_PER_KEPLER_YEAR, ecc=ecc,
                               tau=tau, epoch_ref_mjd=EPOCH_REF_MJD)
    if not own_zero_points:
        return X
    column_b = (instrument_number == 1).astype(float)     # 1 on the rows of "B", 0 elsewhere
    column_c = (instrument_number == 2).astype(float)     # 1 on the rows of "C", 0 elsewhere
    return concat_columns([X, np.column_stack([column_b, column_c])])


X_true = design_matrix(4000.0, 0.70, 0.35, own_zero_points=True)
print("shape of the design matrix:", X_true.shape, " columns: gamma, C, S, offset of B, offset of C")
rows_to_show = [0, 16, 17, 41, 42, 53]          # the first and the last row of each instrument
for row in rows_to_show:
    print(f"row {row:2d} ({NAMES[instrument_number[row]]:4s}):", np.round(X_true[row], 3))

The last two columns say which instrument a row belongs to. A `gaia` row has `0 0`: its zero point is $\gamma$. A `B` row has `1 0`: its zero point is $\gamma$ plus the fourth amplitude.

## 3. Refine the shape, then read the zero points

At each shape $(P, e, \tau)$ one linear solve gives the five amplitudes and the score `logL_marginal`. Nelder–Mead maximises the score over the shape. **Refine the shape before you read the zero points**: on a coarse grid the 0/1 columns take up part of the shape error.

In [ ]:
def solve_at(shape, own_zero_points):
    # One linear solve at a shape (P in days, e, tau).
    period_days, ecc, tau = shape
    X = design_matrix(period_days, ecc, tau % 1.0, own_zero_points)   # tau % 1.0: tau is a phase
    return linear_solve_rv(rv_all, sigma_all, X)


def negative_score(shape, own_zero_points):
    # Minus the marginal likelihood; a wall outside the allowed shapes.
    period_days, ecc = shape[0], shape[1]
    if not (500.0 < period_days < 30000.0 and 0.0 <= ecc < 0.95):
        return np.inf
    try:
        return -solve_at(shape, own_zero_points).logL_marginal
    except ValueError:                          # a singular design at this shape
        return np.inf


def refine(own_zero_points):
    # Nelder-Mead from a start 3 % off in period; returns the shape and the solve at it.
    start = np.array([3885.0, 0.65, 0.33])
    result = minimize(negative_score, start, args=(own_zero_points,), method="Nelder-Mead",
                      options=dict(xatol=1e-6, fatol=1e-9, maxiter=20000))
    return result.x, solve_at(result.x, own_zero_points)


shape_own, solution_own = refine(own_zero_points=True)
errors_own = np.sqrt(np.diag(solution_own.cov))
true_offset_b = TRUE_ZERO_POINT["B"] - TRUE_ZERO_POINT["gaia"]
true_offset_c = TRUE_ZERO_POINT["C"] - TRUE_ZERO_POINT["gaia"]
print(f"P = {shape_own[0]:.0f} d (true 4000), e = {shape_own[1]:.3f} (true 0.700), "
      f"chi2 / dof = {solution_own.chi2 / solution_own.dof:.2f}")
print(f"K = {recover_K(solution_own.beta):.2f} km/s, omega = {recover_omega(solution_own.beta):.3f} rad (true 1.200)")
# beta[0] is gamma; beta[3] and beta[4] are the amplitudes of the two 0/1 columns
print(f"zero point of gaia : {solution_own.beta[0]:6.2f} +/- {errors_own[0]:.2f} km/s (true {TRUE_ZERO_POINT['gaia']:.2f})")
print(f"B minus gaia       : {solution_own.beta[3]:+6.2f} +/- {errors_own[3]:.2f} km/s (true {true_offset_b:+.2f})")
print(f"C minus gaia       : {solution_own.beta[4]:+6.2f} +/- {errors_own[4]:.2f} km/s (true {true_offset_c:+.2f})")

`recover_K` and `recover_omega` read columns 1 and 2 only, so the extra columns do not disturb them.

## 4. The wrong model: one zero point for all

Fit the same data with the three columns only.

In [ ]:
shape_one, solution_one = refine(own_zero_points=False)
print(f"own zero points: chi2 / dof = {solution_own.chi2:.1f} / {solution_own.dof} = {solution_own.chi2 / solution_own.dof:.2f}")
print(f"one zero point : chi2 / dof = {solution_one.chi2:.1f} / {solution_one.dof} = {solution_one.chi2 / solution_one.dof:.2f}")
print(f"P with one zero point: {shape_one[0]:.0f} d, e = {shape_one[1]:.3f}")

The fit with one zero point is clearly worse. Here the orbit is still found, because instrument `B` alone covers most of it. Do not count on that: a zero-point difference between two instruments that do not overlap in time looks like orbital motion.

## 5. The full likelihood: one `loglike` call per instrument, added

The linear layer holds the jitter fixed. To fit the jitters too, write the likelihood. `loglike` takes one zero point and one jitter, so the likelihood of the three instruments is a sum of three calls. First check that the sum is the Gaussian likelihood written by hand.

In [ ]:
def summed_loglike(orbit, zero_point, jitter):
    # The log-likelihood of all instruments: one orblet loglike call per instrument.
    total = 0.0
    for name in NAMES:
        total += loglike(DATA[name]["t"], DATA[name]["rv"], DATA[name]["err"],
                         offset_kms=zero_point[name], jitter_kms=jitter[name],
                         epoch_ref_mjd=EPOCH_REF_MJD, **orbit)
    return total


by_orblet = summed_loglike(TRUE_ORBIT, TRUE_ZERO_POINT, TRUE_JITTER)
by_hand = 0.0
for name in NAMES:
    model = rv_model(DATA[name]["t"], offset_kms=TRUE_ZERO_POINT[name],
                     epoch_ref_mjd=EPOCH_REF_MJD, **TRUE_ORBIT)
    variance = DATA[name]["err"] ** 2 + TRUE_JITTER[name] ** 2
    by_hand += -0.5 * np.sum((DATA[name]["rv"] - model) ** 2 / variance + np.log(2.0 * np.pi * variance))
print(f"sum of loglike = {by_orblet:.10f}; by hand = {by_hand:.10f}; difference = {by_orblet - by_hand:.1e}")

Now maximise it over eleven numbers: the orbit ($P$, $e$, $\omega$, $\tau$, the companion mass), three zero points and three jitters. The jitters enter as logarithms, so they stay positive. The start is the linear solution of section 3 with small jitters.

In [ ]:
MASS_DIFFERENCE = TRUE_ORBIT["M_msun"] - TRUE_ORBIT["mass_msun"]   # primary mass, held at its true value


def unpack(vector):
    # The eleven numbers -> the orbit keywords, the zero points and the jitters.
    companion_mass = vector[4]
    orbit = dict(period_yr=vector[0], ecc=vector[1], omega_rad=vector[2], tau=vector[3],
                 mass_msun=companion_mass, M_msun=MASS_DIFFERENCE + companion_mass)
    zero_point = dict(zip(NAMES, vector[5:8]))            # entries 5 to 7: the three zero points
    jitter = dict(zip(NAMES, np.exp(vector[8:11])))       # entries 8 to 10: the log jitters
    return orbit, zero_point, jitter


def negative_loglike(vector):
    inside = (vector[0] > 0.0) and (0.0 <= vector[1] < 0.95) and (vector[4] > 0.0)
    if not inside:
        return np.inf
    orbit, zero_point, jitter = unpack(vector)
    value = summed_loglike(orbit, zero_point, jitter)
    if not np.isfinite(value):
        return np.inf
    return -value


gamma_start = solution_own.beta[0]
start = np.array([shape_own[0] / DAYS_PER_KEPLER_YEAR, shape_own[1], recover_omega(solution_own.beta),
                  shape_own[2] % 1.0, 28.0,
                  gamma_start, gamma_start + solution_own.beta[3], gamma_start + solution_own.beta[4],
                  np.log(0.1), np.log(0.1), np.log(0.1)])
options = dict(maxiter=40000, maxfev=40000, xatol=1e-7, fatol=1e-9)
result = minimize(negative_loglike, start, method="Nelder-Mead", options=options)
result = minimize(negative_loglike, result.x, method="Nelder-Mead", options=options)   # a restart
orbit_fit, zero_point_fit, jitter_fit = unpack(result.x)
print(f"P = {orbit_fit['period_yr'] * DAYS_PER_KEPLER_YEAR:.0f} d (true 4000), e = {orbit_fit['ecc']:.3f} (true 0.700)")
print(f"log-likelihood at the maximum {-result.fun:.2f}, at the truth {by_orblet:.2f}")
for name in NAMES:
    print(f"{name:5s} zero point {zero_point_fit[name]:6.2f} km/s (true {TRUE_ZERO_POINT[name]:5.2f}); "
          f"jitter {jitter_fit[name]:5.2f} km/s (true {TRUE_JITTER[name]:4.2f}); {DATA[name]['t'].size} rows")

## What to remember

1. One 0/1 column per instrument other than the reference. $\gamma$ is the zero point of the reference; each extra amplitude is a difference from it, with its error from the covariance.
2. The jitter of an instrument goes into the errors of its rows. The linear layer holds it fixed; the full likelihood fits it.
3. The likelihood of several instruments is a sum of `loglike` calls.
4. Refine the shape before you read the zero points.

Limits of this example: one orbit and one noise draw; a local optimiser that starts near the answer; no sampler, so the jitters and the eleven-number fit have no error bars here. With 12 to 25 rows per instrument the jitters are weakly constrained. To sample, give the summed likelihood to your sampler, as `fit_rv_orbit_all_parameters.ipynb` does for one instrument.

`best_linear_params_rv`, `compose_rv_seed` and `loglike_joint` take one instrument only.